# sEMG Prosthetic Hand — End-to-End Simulation

**Notebook 02 · `02_e2e_simulation.ipynb`**

Simulates the complete sEMG → gesture → servo pipeline **without hardware**:

| Stage | Description |
|-------|-------------|
| 1 | Synthetic sEMG signal generation (4 ch, realistic burst+noise) |
| 1b | Real NinaPro DB5 signal excerpt for comparison |
| 2 | AFE analog filter simulation (HPF 20 Hz · Notch 50 Hz · LPF 100 Hz) |
| 3 | Feature extraction — MAV, RMS, WL, ZC, SSC, VAR per channel (24 features) |
| 4 | INT8 TFLite model inference (real `semg_v3_int8.tflite`) |
| 5 | Map classification → servo angles (config.h `GESTURE_POSTURES[8][6]`) |
| 6 | Animated servo-angle bar chart |
| 7 | Simulated BLE byte packet (ESP32 UART/BLE payload) |


## Cell 1 — Imports & Path Setup

Load all dependencies and verify paths to the model and norm-stats files.

In [ ]:
import os, sys, struct, warnings
warnings.filterwarnings('ignore')

import numpy as np
import scipy.io
import scipy.signal as signal
import matplotlib
matplotlib.use('Agg')          # non-interactive; switch to 'inline' in JupyterLab
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import display, HTML

# ── TFLite (use the bundled tf.lite first, fall back to tflite_runtime) ───────
try:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter
except Exception:
    import tflite_runtime.interpreter as tflite
    Interpreter = tflite.Interpreter

# ── Project paths ─────────────────────────────────────────────────────────────
BASE = "/home/kiarie/school/Final Year project"
MODEL_PATH   = os.path.join(BASE, "firmware/model/semg_v3_int8.tflite")
MEAN_PATH    = os.path.join(BASE, "ml/models/feature_mean_v3.npy")
STD_PATH     = os.path.join(BASE, "ml/models/feature_std_v3.npy")
NINAPRO_PATH = os.path.join(BASE, "ml/data/ninapro_db5/S1/s1/S1_E1_A1.mat")

for p in [MODEL_PATH, MEAN_PATH, STD_PATH, NINAPRO_PATH]:
    tag = "✓" if os.path.exists(p) else "✗ MISSING"
    print(f"{tag}  {os.path.relpath(p, BASE)}")

# ── Constants ─────────────────────────────────────────────────────────────────
FS_HW   = 2000    # hardware sampling rate (Hz)
FS_DB5  = 200     # NinaPro DB5 sampling rate (Hz)
WIN_HW  = 500     # 250 ms window at 2000 Hz
WIN_DB5 = 50      # 250 ms window at 200 Hz
CHANNELS = [0, 4, 8, 12]   # 4 of 16 NinaPro channels

GESTURE_NAMES = ["Rest","Open","Power","Pinch","Point","WristFlex","WristExt","ThumbsUp"]
# GESTURE_POSTURES[8][6] — angles (deg): thumb,index,middle,ring,pinky,wrist
GESTURE_POSTURES = np.array([
    [  0,  0,  0,  0,  0, 90],   # 0 Rest
    [  0,  0,  0,  0,  0, 90],   # 1 Open
    [180,180,180,180,180, 90],   # 2 Power
    [  0,180,180,180,180, 90],   # 3 Pinch
    [180,  0,180,180,180, 90],   # 4 Point
    [  0,  0,  0,  0,  0, 45],   # 5 WristFlex
    [  0,  0,  0,  0,  0,135],   # 6 WristExt
    [  0,180,180,180,180, 90],   # 7 ThumbsUp
], dtype=np.float32)

SERVO_NAMES = ["Thumb","Index","Middle","Ring","Pinky","Wrist"]
print("\nConstants OK. TF/TFLite loaded.")

## Stage 1 — Synthetic sEMG Signal Generation

Generates a 4-channel, 250 ms sEMG burst at **2000 Hz** (hardware rate) that mimics:
- Background white noise floor (~±5 ADC counts)
- Burst of activity (bandpass Gaussian noise 20–500 Hz, amplitude ~±60 ADC counts)
- 50 Hz mains interference (single tone)
- DC offset (electrode half-cell potential)


In [ ]:
rng = np.random.default_rng(42)

def synthetic_semg(n_samples=WIN_HW, fs=FS_HW, n_ch=4, seed=42):
    """Return (n_samples, n_ch) int16 synthetic sEMG in ADC counts (8-bit Myo range)."""
    rng = np.random.default_rng(seed)
    t   = np.arange(n_samples) / fs
    out = np.zeros((n_samples, n_ch), dtype=np.float32)

    for ch in range(n_ch):
        # Background noise
        noise = rng.normal(0, 3, n_samples).astype(np.float32)
        # Burst: bandpass Gaussian noise
        raw_burst = rng.normal(0, 40, n_samples).astype(np.float32)
        sos = signal.butter(4, [20/(fs/2), 450/(fs/2)], btype='band', output='sos')
        burst = signal.sosfilt(sos, raw_burst).astype(np.float32)
        # Envelope shaping (raised cosine ramp)
        env = np.zeros(n_samples, dtype=np.float32)
        s, e = int(0.05*n_samples), int(0.90*n_samples)
        ramp = int(0.1 * (e - s))
        env[s:s+ramp]   = np.linspace(0, 1, ramp)
        env[s+ramp:e-ramp] = 1.0
        env[e-ramp:e]   = np.linspace(1, 0, ramp)
        # 50 Hz mains
        mains = (2.0 * np.sin(2 * np.pi * 50 * t)).astype(np.float32)
        # DC offset (varies per channel)
        dc = float(rng.uniform(-8, 8))
        out[:, ch] = dc + noise + burst * env + mains

    return np.clip(out, -128, 127).astype(np.float32)

synth = synthetic_semg()
t_hw  = np.arange(WIN_HW) / FS_HW * 1000   # ms

fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
ch_labels = [f"CH{i+1} (idx {CHANNELS[i]})" for i in range(4)]
for i, ax in enumerate(axes):
    ax.plot(t_hw, synth[:, i], lw=0.7, color=f"C{i}")
    ax.set_ylabel(ch_labels[i], fontsize=8)
    ax.set_ylim(-140, 140)
    ax.axhline(0, color='k', lw=0.3)
axes[-1].set_xlabel("Time (ms)")
axes[0].set_title("Stage 1 — Synthetic sEMG (4 channels, 250 ms, 2 kHz)")
plt.tight_layout()
plt.savefig("/tmp/stage1_synthetic.png", dpi=100)
plt.show()
print(f"Synthetic signal shape: {synth.shape}  range: [{synth.min():.1f}, {synth.max():.1f}]")

## Stage 1b — Real NinaPro DB5 Signal (S1, Exercise 1)

Loads `S1_E1_A1.mat` and extracts a representative **250 ms window** for each gesture.
Key details:
- DB5 sampled at **200 Hz** → 50-sample windows
- Labels in `restimulus` (not `stimulus`)
- 4 channels selected: indices `[0, 4, 8, 12]`


In [ ]:
mat = scipy.io.loadmat(NINAPRO_PATH)
emg_full   = mat['emg'].astype(np.float32)          # (N, 16)
labels_full = mat['restimulus'].squeeze().astype(int) # (N,)

emg_4ch = emg_full[:, CHANNELS]    # (N, 4)
print(f"EMG matrix: {emg_full.shape}  |  4-channel: {emg_4ch.shape}")
print(f"Unique gesture labels: {np.unique(labels_full)}")
print(f"Label counts: { {int(g): int(np.sum(labels_full==g)) for g in np.unique(labels_full)} }")

# Pick a clean 50-sample window for each non-rest gesture
fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
t_db5 = np.arange(WIN_DB5) / FS_DB5 * 1000

# Use gesture 2 (Power) as the demo window
for g_demo in range(1, 8):
    idx = np.where(labels_full == g_demo)[0]
    if len(idx) >= WIN_DB5:
        win_start = idx[len(idx)//2]
        demo_win  = emg_4ch[win_start : win_start + WIN_DB5]
        if demo_win.shape[0] == WIN_DB5:
            break

for i, ax in enumerate(axes):
    ax.plot(t_db5, demo_win[:, i], lw=1.0, color=f"C{i}")
    ax.set_ylabel(f"CH{i+1}", fontsize=9)
    ax.axhline(0, color='k', lw=0.3)
axes[-1].set_xlabel("Time (ms)")
axes[0].set_title(f"Stage 1b — NinaPro DB5 Real Signal (gesture: {GESTURE_NAMES[g_demo]}, 250 ms, 200 Hz)")
plt.tight_layout()
plt.savefig("/tmp/stage1b_ninapro.png", dpi=100)
plt.show()
print(f"Demo window: gesture={g_demo} ({GESTURE_NAMES[g_demo]}), start={win_start}")

## Stage 2 — AFE Analog Filter Simulation

Models the hardware Analog Front-End filter chain:

| Filter | Type | Parameters |
|--------|------|------------|
| HPF | 2nd-order Butterworth | fc = 20 Hz |
| Notch | IIR notch | f0 = 50 Hz, Q = 30 |
| LPF | 4th-order Butterworth | fc = 100 Hz |

Applied via `scipy.signal.sosfilt` (forward-only, matches real-time embedded implementation).


In [ ]:
def build_afe_filters(fs=FS_HW):
    # Returns dict of SOS filter coefficients for the AFE chain.
    nyq = fs / 2
    hpf   = signal.butter(2, 20/nyq,        btype='high', output='sos')
    notch = signal.iirnotch(50, Q=30, fs=fs)
    notch_sos = signal.tf2sos(*notch)
    lpf   = signal.butter(4, 100/nyq,        btype='low',  output='sos')
    return {'hpf': hpf, 'notch': notch_sos, 'lpf': lpf}

def apply_afe(sig, fs=FS_HW):
    # Apply HPF -> Notch -> LPF to (N, n_ch) signal. Returns dict of stage outputs.
    filt = build_afe_filters(fs)
    after_hpf   = signal.sosfilt(filt['hpf'],   sig, axis=0).astype(np.float32)
    after_notch = signal.sosfilt(filt['notch'],  after_hpf,   axis=0).astype(np.float32)
    after_lpf   = signal.sosfilt(filt['lpf'],    after_notch, axis=0).astype(np.float32)
    return {'raw': sig, 'hpf': after_hpf, 'notch': after_notch, 'lpf': after_lpf}

afe_stages = apply_afe(synth, fs=FS_HW)
filtered   = afe_stages['lpf']   # final output used downstream

# ── Plot 4-panel filter progression for CH1 ──────────────────────────────────
stage_labels = ['Raw', 'After HPF (20 Hz)', 'After Notch (50 Hz)', 'After LPF (100 Hz)']
stage_keys   = ['raw', 'hpf', 'notch', 'lpf']
colors       = ['#888888', '#e07b00', '#c90000', '#1f77b4']

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, key, label, col in zip(axes, stage_keys, stage_labels, colors):
    ax.plot(t_hw, afe_stages[key][:, 0], lw=0.7, color=col)
    ax.set_ylabel(label, fontsize=8)
    ax.axhline(0, color='k', lw=0.3)
axes[-1].set_xlabel("Time (ms)")
axes[0].set_title("Stage 2 — AFE Filter Chain (CH1 shown)")
plt.tight_layout()
plt.savefig("/tmp/stage2_afe.png", dpi=100)
plt.show()

# ── Bode plots ────────────────────────────────────────────────────────────────
filt = build_afe_filters(FS_HW)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
filter_items = [('HPF 20 Hz', filt['hpf']), ('Notch 50 Hz', filt['notch']),
                ('LPF 100 Hz', filt['lpf'])]
for ax, (name, sos) in zip(axes, filter_items):
    w, h = signal.sosfreqz(sos, worN=4096, fs=FS_HW)
    ax.plot(w, 20*np.log10(np.abs(h)+1e-12), lw=1.2)
    ax.set_xlim(0, 500); ax.set_ylim(-60, 5)
    ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Magnitude (dB)")
    ax.set_title(f"Bode — {name}"); ax.grid(True, alpha=0.3)
    ax.axvline(50, color='r', lw=0.8, ls='--', label='50 Hz mains')
plt.tight_layout()
plt.savefig("/tmp/stage2_bode.png", dpi=100)
plt.show()
print("AFE filters applied. Final filtered shape:", filtered.shape)

## Stage 3 — Feature Extraction (24 features)

Extracts **6 time-domain features per channel × 4 channels = 24 features** from the 500-sample hardware window:

| Feature | Formula |
|---------|--------|
| MAV | `mean(|x|)` |
| RMS | `sqrt(mean(x²))` |
| WL  | `sum(|x[n]-x[n-1]|)` |
| ZC  | `count(sign changes)` |
| SSC | `count(slope sign changes)` |
| VAR | `var(x)` |

Feature vector layout: `[MAV_0, RMS_0, WL_0, ZC_0, SSC_0, VAR_0, ..., VAR_3]` (24 elements)


In [ ]:
def extract_features(win):
    # win: (N, 4) float32. Returns (24,) float32 feature vector."
    feats = []
    for ch in range(win.shape[1]):
        x = win[:, ch]
        mav = float(np.mean(np.abs(x)))
        rms = float(np.sqrt(np.mean(x**2)))
        wl  = float(np.sum(np.abs(np.diff(x))))
        zc  = float(np.sum(np.diff(np.sign(x)) != 0))
        ssc = float(np.sum(np.diff(np.sign(np.diff(x))) != 0))
        var = float(np.var(x))
        feats.extend([mav, rms, wl, zc, ssc, var])
    return np.array(feats, dtype=np.float32)

feat_names_1ch = ["MAV","RMS","WL","ZC","SSC","VAR"]
feat_labels = [f"{f}_CH{c+1}" for c in range(4) for f in feat_names_1ch]

# Extract from the synthetic (hardware-rate) filtered signal
features_raw = extract_features(filtered)

# ── Normalize with training stats ─────────────────────────────────────────────
feat_mean = np.load(MEAN_PATH)
feat_std  = np.load(STD_PATH)
feat_std  = np.where(feat_std < 1e-8, 1.0, feat_std)
features_norm = (features_raw - feat_mean) / feat_std

print(f"Raw feature vector (24,): {features_raw[:6]} ...")
print(f"Norm feature vector (24,): {features_norm[:6]} ...")

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))
bar_colors = [f"C{i//6}" for i in range(24)]
ax1.bar(range(24), features_raw, color=bar_colors, edgecolor='k', linewidth=0.3)
ax1.set_xticks(range(24)); ax1.set_xticklabels(feat_labels, rotation=45, ha='right', fontsize=7)
ax1.set_ylabel("Raw value"); ax1.set_title("Stage 3 — Raw Feature Vector (24 features)")

ax2.bar(range(24), features_norm, color=bar_colors, edgecolor='k', linewidth=0.3)
ax2.set_xticks(range(24)); ax2.set_xticklabels(feat_labels, rotation=45, ha='right', fontsize=7)
ax2.axhline(0, color='k', lw=0.5)
ax2.set_ylabel("Normalised (z-score)"); ax2.set_title("Stage 3 — Normalised Feature Vector")
plt.tight_layout()
plt.savefig("/tmp/stage3_features.png", dpi=100)
plt.show()
print(f"Feature extraction OK. Shape: {features_norm.shape}")

## Stage 4 — INT8 TFLite Model Inference

Runs the real quantised model (`semg_v3_int8.tflite`) on the normalised feature vector.

**Quantisation protocol:**
- Input: float32 → INT8 via `input_scale=0.04123, input_zero_point=-21`
- Output: INT8 → float32 (softmax probabilities) via `output_scale=0.00390625, output_zero_point=-128`


In [ ]:
# Load the interpreter once
interp = Interpreter(MODEL_PATH)
interp.allocate_tensors()
inp_detail = interp.get_input_details()[0]
out_detail = interp.get_output_details()[0]

in_scale  = inp_detail['quantization'][0]
in_zp     = inp_detail['quantization'][1]
out_scale = out_detail['quantization'][0]
out_zp    = out_detail['quantization'][1]

print(f"Model input  : shape={inp_detail['shape']}  dtype={inp_detail['dtype'].__name__}")
print(f"  quantization: scale={in_scale:.6f}, zero_point={in_zp}")
print(f"Model output : shape={out_detail['shape']}  dtype={out_detail['dtype'].__name__}")
print(f"  quantization: scale={out_scale:.6f}, zero_point={out_zp}")

def run_inference(feat_norm):
    # Run INT8 TFLite inference. Returns (8,) softmax probabilities."
    # float32 → int8 quantisation
    x_q = np.round(feat_norm / in_scale + in_zp).clip(-128, 127).astype(np.int8)
    interp.set_tensor(inp_detail['index'], x_q.reshape(1, 24))
    interp.invoke()
    y_q = interp.get_tensor(out_detail['index'])[0]    # (8,) int8
    # int8 → float32 dequantisation
    probs = (y_q.astype(np.float32) - out_zp) * out_scale
    # Clip negatives (quantisation artefacts) and re-normalise
    probs = np.maximum(probs, 0.0)
    s = probs.sum()
    if s > 0:
        probs /= s
    return probs

probs = run_inference(features_norm)
pred  = int(np.argmax(probs))
conf  = float(probs[pred])

print(f"\nSynthetic signal prediction: class {pred} ({GESTURE_NAMES[pred]})  confidence {conf:.3f}")
print(f"Full probability vector: { {GESTURE_NAMES[i]: f'{probs[i]:.3f}' for i in range(8)} }")

# ── Confidence bar chart ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
bar_cols = ['#1f77b4' if i != pred else '#d62728' for i in range(8)]
ax.bar(GESTURE_NAMES, probs, color=bar_cols, edgecolor='k', linewidth=0.4)
ax.set_ylabel("Probability"); ax.set_ylim(0, 1)
ax.set_title(f"Stage 4 — Model Confidence (predicted: {GESTURE_NAMES[pred]})")
ax.axhline(0.5, color='gray', ls='--', lw=0.8)
for i, p in enumerate(probs):
    ax.text(i, p + 0.01, f"{p:.2f}", ha='center', fontsize=8)
plt.tight_layout()
plt.savefig("/tmp/stage4_confidence.png", dpi=100)
plt.show()

## Stage 5 — Map Classification → Servo Angles

Looks up the predicted gesture in `GESTURE_POSTURES[8][6]` (from `config.h`).
Each gesture maps to 6 servo angles: `[Thumb, Index, Middle, Ring, Pinky, Wrist]`.


In [ ]:
target_angles = GESTURE_POSTURES[pred]
print(f"Predicted gesture : {pred} — {GESTURE_NAMES[pred]}")
print(f"Servo angles      : { {SERVO_NAMES[i]: int(target_angles[i]) for i in range(6)} }")

fig, ax = plt.subplots(figsize=(9, 4))
bar_col = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']
bars = ax.bar(SERVO_NAMES, target_angles, color=bar_col, edgecolor='k', linewidth=0.4)
ax.set_ylim(0, 200); ax.set_ylabel("Angle (°)")
ax.set_title(f"Stage 5 — Servo Target Angles: {GESTURE_NAMES[pred]}")
ax.axhline(90,  color='gray', ls='--', lw=0.7, label='90° (neutral wrist)')
ax.axhline(180, color='red',  ls='--', lw=0.7, label='180° (full flex)')
for bar, angle in zip(bars, target_angles):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f"{int(angle)}°", ha='center', fontsize=10, fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("/tmp/stage5_servos.png", dpi=100)
plt.show()

## Stage 6 — Animated Servo Motion

Runs the full pipeline for **all 8 gestures in sequence** and animates the servo bar chart.
Each frame uses a freshly generated synthetic burst matched to a gesture class.

> 💡 The animation is saved as `/tmp/stage6_servo_animation.gif` and displayed inline.


In [ ]:
import matplotlib.patches as mpatches

def pipeline_for_gesture(gesture_idx, seed=0):
    # Synthetic signal -> filter -> features -> TFLite -> servo angles.
    # Generate slightly stronger signal for specific gesture
    s   = synthetic_semg(seed=seed + gesture_idx * 7)
    afe = apply_afe(s, fs=FS_HW)
    flt = afe['lpf']
    fraw = extract_features(flt)
    fnorm = (fraw - feat_mean) / feat_std
    p    = run_inference(fnorm)
    ppred = int(np.argmax(p))
    angles = GESTURE_POSTURES[gesture_idx]    # ground-truth angles
    return angles, ppred, p

# ── Pre-compute all 8 gesture results ─────────────────────────────────────────
results = [pipeline_for_gesture(g, seed=42) for g in range(8)]

# ── Build animation ────────────────────────────────────────────────────────────
fig_anim, ax_anim = plt.subplots(figsize=(9, 5))
plt.subplots_adjust(bottom=0.18, top=0.85)

x = np.arange(6)
bar_col = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']

# Initial frame
angles0, pred0, prob0 = results[0]
bars_anim = ax_anim.bar(x, angles0, color=bar_col, edgecolor='k', linewidth=0.4)
ax_anim.set_xticks(x); ax_anim.set_xticklabels(SERVO_NAMES)
ax_anim.set_ylim(0, 200); ax_anim.set_ylabel("Angle (°)")
ax_anim.axhline(90, color='gray', ls='--', lw=0.7)
title_txt = ax_anim.set_title("", fontsize=12)

angle_texts = [ax_anim.text(i, 0, '', ha='center', fontsize=9, fontweight='bold')
               for i in range(6)]

def update_anim(frame):
    g = frame % 8
    angles, ppred, p = results[g]
    for bar, ang, txt in zip(bars_anim, angles, angle_texts):
        bar.set_height(ang)
        txt.set_position((bar.get_x() + bar.get_width()/2, ang + 3))
        txt.set_text(f"{int(ang)}°")
    match_str = "✓" if ppred == g else f"→{GESTURE_NAMES[ppred]}"
    title_txt.set_text(
        f"Gesture {g}: {GESTURE_NAMES[g]}  |  Model: {GESTURE_NAMES[ppred]} {match_str}  "
        f"(conf {p[ppred]:.2f})"
    )
    return bars_anim

anim = animation.FuncAnimation(fig_anim, update_anim, frames=16,
                                interval=800, blit=False)

gif_path = "/tmp/stage6_servo_animation.gif"
writer = animation.PillowWriter(fps=1.2)
anim.save(gif_path, writer=writer)
plt.close(fig_anim)

from IPython.display import Image as IPyImage
display(IPyImage(gif_path))
print(f"Animation saved: {gif_path}")
print("\nGesture sweep results:")
print(f"{'Gesture':<12} {'GT Angles':>30}  Model prediction")
for g, (angles, ppred, p) in enumerate(results):
    angle_str = str([int(a) for a in angles])
    match = "✓" if ppred == g else f"✗ ({GESTURE_NAMES[ppred]})"
    print(f"  {g} {GESTURE_NAMES[g]:<10} {angle_str:>32}  {match}  conf={p[ppred]:.2f}")

## Stage 7 — Simulated BLE Packet

Formats the byte payload the ESP32 would transmit over BLE GATT Notify.

**Packet format (9 bytes):**

```
[ 0xAA ]  start byte
[ gesture_id : uint8 ]  0-7
[ servo0_angle : uint8 ]  Thumb  (0-180°)
[ servo1_angle : uint8 ]  Index
[ servo2_angle : uint8 ]  Middle
[ servo3_angle : uint8 ]  Ring
[ servo4_angle : uint8 ]  Pinky
[ servo5_angle : uint8 ]  Wrist
[ checksum : uint8 ]  XOR of bytes 1-7
```


In [ ]:
def build_ble_packet(gesture_id, servo_angles):
    # Build 9-byte BLE packet. Returns bytes object and hex string."
    start = 0xAA
    payload = [int(gesture_id)] + [int(a) for a in servo_angles]
    checksum = 0
    for b in payload:
        checksum ^= b
    packet = bytes([start] + payload + [checksum])
    return packet

def decode_ble_packet(pkt):
    # Decode and validate a BLE packet."
    assert pkt[0] == 0xAA, "Bad start byte"
    checksum_rx = pkt[-1]
    checksum_calc = 0
    for b in pkt[1:-1]:
        checksum_calc ^= b
    valid = (checksum_rx == checksum_calc)
    g_id  = pkt[1]
    angles = list(pkt[2:8])
    return g_id, angles, valid

# ── Simulate a BLE transmission for each gesture ──────────────────────────────
print("=" * 65)
print(f"{'BLE Packet Simulation — ESP32 GATT Notify Characteristic':^65}")
print("=" * 65)
print(f"{'Byte':>5}  {'Field':<18}  {'Value'}")
print("-" * 65)

# Use the synthetic signal prediction
pkt = build_ble_packet(pred, target_angles)
field_names = ["START_BYTE","GESTURE_ID","SERVO_THUMB","SERVO_INDEX",
               "SERVO_MIDDLE","SERVO_RING","SERVO_PINKY","SERVO_WRIST","CHECKSUM"]
for i, (b, name) in enumerate(zip(pkt, field_names)):
    extra = f"  ← {GESTURE_NAMES[pred]}" if name == "GESTURE_ID" else (
            f"  ← {int(b)}°" if "SERVO" in name else "")
    print(f"  [{i}]  {name:<18}  0x{b:02X}  ({int(b):3d}){extra}")
print("-" * 65)
print(f"Full packet hex: {pkt.hex(' ').upper()}")
print(f"Packet length  : {len(pkt)} bytes")

# Decode and verify
g_dec, angles_dec, valid = decode_ble_packet(pkt)
print(f"\nDecode check   : gesture={g_dec} ({GESTURE_NAMES[g_dec]})  angles={angles_dec}  CRC={'OK ✓' if valid else 'FAIL ✗'}")

# ── Print all 8 gestures ──────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("All 8 gesture packets:")
print(f"{'Gesture':<12} {'Hex payload':>52}")
print("-" * 65)
for g in range(8):
    pkt_g = build_ble_packet(g, GESTURE_POSTURES[g])
    print(f"  {g} {GESTURE_NAMES[g]:<10}  {pkt_g.hex(' ').upper()}")
print("=" * 65)

## Bonus — NinaPro Real-Signal Inference Demo

Extracts one 50-sample window per gesture from the real NinaPro DB5 data,
upsamples to 2 kHz for hardware-rate processing, runs the full pipeline,
and prints predictions vs ground truth.


In [ ]:
from scipy.signal import resample

print(f"{'Gesture':<12} {'GT':>3} {'Pred':>12} {'Conf':>6}  {'Match'}")
print("-" * 55)

# We'll collect results for a confusion-style summary
gt_list, pred_list = [], []

for g_id in range(8):
    idx = np.where(labels_full == g_id)[0]
    if len(idx) < WIN_DB5:
        print(f"  {g_id} {GESTURE_NAMES[g_id]:<10}  — not enough samples in S1_E1_A1 —")
        continue
    # Use the middle of the gesture segment
    mid = idx[len(idx)//2]
    win_db5 = emg_4ch[mid : mid + WIN_DB5]
    if win_db5.shape[0] < WIN_DB5:
        continue

    # Upsample DB5 (200 Hz) → hardware rate (2000 Hz) for the filter chain
    win_hw = resample(win_db5, WIN_HW).astype(np.float32)

    # Run the pipeline
    afe_out  = apply_afe(win_hw, fs=FS_HW)
    feat_raw = extract_features(afe_out['lpf'])
    feat_n   = (feat_raw - feat_mean) / feat_std
    probs_g  = run_inference(feat_n)
    ppred_g  = int(np.argmax(probs_g))
    conf_g   = float(probs_g[ppred_g])

    match = "✓" if ppred_g == g_id else "✗"
    print(f"  {g_id} {GESTURE_NAMES[g_id]:<10}  →  {GESTURE_NAMES[ppred_g]:<12} {conf_g:.3f}  {match}")
    gt_list.append(g_id); pred_list.append(ppred_g)

accuracy = sum(g == p for g, p in zip(gt_list, pred_list)) / len(gt_list) if gt_list else 0
print(f"\nSingle-window accuracy on S1 sample: {accuracy:.2%}  ({sum(g==p for g,p in zip(gt_list,pred_list))}/{len(gt_list)})")
print("\n(Note: cross-subject single-window accuracy is expected ~50-65% without fine-tuning.)")

## Pipeline Summary

| Stage | Component | Details |
|-------|-----------|--------|
| 1 | Signal source | Synthetic sEMG (4 ch, 500 samples, 2 kHz) |
| 1b | Real signal | NinaPro DB5 S1 (200 Hz → 2 kHz upsampled) |
| 2 | AFE filters | HPF 20 Hz + Notch 50 Hz + LPF 100 Hz |
| 3 | Features | 24 time-domain (MAV/RMS/WL/ZC/SSC/VAR × 4 ch) |
| 4 | Inference | `semg_v3_int8.tflite` INT8 MLP (38 KB) |
| 5 | Servo map | `GESTURE_POSTURES[8][6]` from config.h |
| 6 | Animation | Matplotlib bar chart (GIF) |
| 7 | BLE packet | 9-byte XOR-checked GATT payload |

**End-to-end latency estimate (embedded):**
- ADC DMA capture: 250 ms
- Feature extraction: ~0.5 ms
- TFLite inference: ~2–4 ms (ESP32-S3)
- Servo PWM update: <1 ms
- **Total decision latency: ~256 ms**


In [ ]:
print("Notebook complete.")
print(f"Model: {os.path.basename(MODEL_PATH)}")
print(f"Norm stats: mean={feat_mean[:3]} ... std={feat_std[:3]} ...")
print(f"Gestures: {GESTURE_NAMES}")
print("All stages executed successfully.")